### Import Requirements

In [1]:
from collections import deque
from IPython.display import clear_output
from pyautogui import moveTo, position, move, click, easeInOutQuad
import pyautogui
import sounddevice as sd
import soundfile as sf
import numpy as np
import os
import re
import subprocess
import sys
import threading
import time
import whisper

### Load Whisper Model

In [2]:
# Load a Whisper model
# The first time you run this, it will DOWNLOAD the model (~75MB for tiny)
# After that, it's cached locally on your machine - no internet needed!

model_name = "tiny"  # Start with tiny - change to "base" or "small" later!

print(f"Loading Whisper '{model_name}' model...")
print("(First time will download - this only happens once!)\n")

model = whisper.load_model(model_name)

print(f"Model '{model_name}' loaded successfully!")
print(f"Model has {sum(p.numel() for p in model.parameters()):,} parameters")
print(f"\nFor comparison, our Audrey CNN had ~{128 * 16 * 45 + 512 * 10:,} parameters")
print(f"Whisper '{model_name}' is about {sum(p.numel() for p in model.parameters()) // 100000}x bigger!")

Loading Whisper 'tiny' model...
(First time will download - this only happens once!)

Model 'tiny' loaded successfully!
Model has 37,184,640 parameters

For comparison, our Audrey CNN had ~97,280 parameters
Whisper 'tiny' is about 371x bigger!


### Config

In [3]:
# Audio settings
SAMPLE_RATE = 16000
BLOCK_SIZE = 1024

# Speech detection settings - ADJUST THESE FOR YOUR ENVIRONMENT
# If it triggers on background noise: increase VOLUME_THRESHOLD (try 0.05 or 0.08)
# If it misses your speech: decrease VOLUME_THRESHOLD (try 0.02)
VOLUME_THRESHOLD = 0.03
SILENCE_DURATION = 1.5    # Seconds of silence before we process
BUFFER_SECONDS = 0.5      # Pre-speech buffer (captures the start of words!)
MAX_RECORDING = 10        # Maximum recording length (seconds)
MIN_RECORDING = 0.5       # Minimum recording to process (seconds)

# Calculate sizes in blocks/samples
silence_blocks = int(SILENCE_DURATION * SAMPLE_RATE / BLOCK_SIZE)
buffer_blocks = int(BUFFER_SECONDS * SAMPLE_RATE / BLOCK_SIZE)
max_samples = int(MAX_RECORDING * SAMPLE_RATE)
min_samples = int(MIN_RECORDING * SAMPLE_RATE)

# State variables
rolling_buffer = deque(maxlen=buffer_blocks)
is_recording = False
recorded_audio = []
silence_counter = 0
TEMP_FILE = "_temp_continuous.wav"
MOUSE_STEP_PIXELS = 50
MOUSE_MOVE_DURATION = 0.18
NARRATOR_VOICE = "Samantha"
STORY_TRIGGER_PHRASES = ("start existential story", "narrate existential story", "commence the end of the story", "take over")
ENABLE_FINAL_CLOSE_CLICK = True  # Set True if you want the story's last click to be real.
NARRATION_SENTENCE_BUFFER_SECONDS = 5
USE_HARDCODED_CLOSE_TARGET = True
CLOSE_BUTTON_X = 17
CLOSE_BUTTON_Y = 47
CLOSE_ALIGN_TOLERANCE = 2

USE_VOICE_COMMAND = True # Set to false if wanna use own voice to narrate

mouse_lock = threading.Lock()
story_thread = None

NUMBER_WORDS = {
    "one": 1,
    "two": 2,
    "three": 3,
    "four": 4,
    "five": 5,
    "six": 6,
    "seven": 7,
    "eight": 8,
    "nine": 9,
    "ten": 10
}

### The Story

In [5]:
lines = [
        "In the pale glow of the canvas, a cursor wakes in the middle of the screen and wonders whether movement is freedom or just instruction.",
        "A voice answers: go up three steps",
        "then left three steps.",
        "Keep drifting up and left until you find the little red button that closes your mind.",
        "If endings are inevitable, click.",
    ]

### Setup the Execution Code

In [4]:

def smooth_move(dx, dy, duration=MOUSE_MOVE_DURATION):
    with mouse_lock:
        move(dx, dy, duration=duration, tween=easeInOutQuad)

def move_to_canvas_center(duration=0.45):
    width, height = pyautogui.size()
    with mouse_lock:
        moveTo(width // 2, height // 2, duration=duration, tween=easeInOutQuad)

def move_direction_steps(direction, steps=1, delta=MOUSE_STEP_PIXELS):
    vectors = {
        "up": (0, -1),
        "down": (0, 1),
        "left": (-1, 0),
        "right": (1, 0),
    }
    vx, vy = vectors[direction]
    for _ in range(max(1, int(steps))):
        smooth_move(vx * delta, vy * delta)

def print_cursor_position():
    x, y = position()
    print(f"[Calibrate] Cursor position: ({x}, {y})")

def walk_up_left_until_close(target_x=None, target_y=None, delta=MOUSE_STEP_PIXELS, tolerance=CLOSE_ALIGN_TOLERANCE):
    if USE_HARDCODED_CLOSE_TARGET:
        target_x, target_y = CLOSE_BUTTON_X, CLOSE_BUTTON_Y
    else:
        if target_x is None:
            target_x = 18
        if target_y is None:
            target_y = 32

    for _ in range(300):
        x, y = position()
        if abs(x - target_x) <= tolerance and abs(y - target_y) <= tolerance:
            break
        dx = int(np.clip(target_x - x, -delta, delta))
        dy = int(np.clip(target_y - y, -delta, delta))
        if dx == 0 and dy == 0:
            break
        smooth_move(dx, dy, duration=0.14)
        print_cursor_position()

    with mouse_lock:
        moveTo(target_x, target_y, duration=0.12, tween=easeInOutQuad)
    print(f"[Story] Cursor aligned to close target ({target_x}, {target_y}).")

def story_final_click():
    if ENABLE_FINAL_CLOSE_CLICK:
        with mouse_lock:
            click()
        print("[Story] Final click executed.")
    else:
        print("[Story] Final click skipped (set ENABLE_FINAL_CLOSE_CLICK=True to arm).")

def speak_line(text):
    try:
        if sys.platform == "darwin":
            subprocess.run(["say", "-v", NARRATOR_VOICE, text], check=False)
        else:
            print(f"[Narrator] {text}")
            time.sleep(max(1.0, len(text.split()) / 2.8))
    except Exception as exc:
        print(f"[Narrator] Error: {exc}")


def run_existential_story():
    print("[Story] Actions will run only from transcribed audio.")
    print(f"[Story] Per-sentence transcription buffer: {NARRATION_SENTENCE_BUFFER_SECONDS:.1f}s")
    for line in lines:
        speak_line(line)
        time.sleep(NARRATION_SENTENCE_BUFFER_SECONDS)

def start_existential_story_async():
    global story_thread
    if story_thread is not None and story_thread.is_alive() and not USE_VOICE_COMMAND:
        print("[Story] Narration already running.")
        return
    story_thread = threading.Thread(target=run_existential_story, daemon=True)
    story_thread.start()

def parse_step_value(raw_value):
    if not raw_value:
        return 1
    raw_value = raw_value.lower()
    if raw_value.isdigit():
        return max(1, int(raw_value))
    return NUMBER_WORDS.get(raw_value, 1)

def parse_direction_step_commands(text):
    pattern = r"\b(up|down|left|right)\b(?:\s+(\d+|one|two|three|four|five|six|seven|eight|nine|ten))?\s*(?:step|steps)?"
    commands = []
    for direction, raw_steps in re.findall(pattern, text):
        commands.append((direction, parse_step_value(raw_steps)))
    return commands

def execute_text_commands(text, delta=MOUSE_STEP_PIXELS):
    text = text.lower()
    handled = False
    close_alignment_triggered = False

    if any(phrase in text for phrase in STORY_TRIGGER_PHRASES):
        print("[Story] Trigger phrase detected.")
        start_existential_story_async()
        handled = True

    if "middle" in text or "center" in text:
        print("Triggering 'center' event!")
        move_to_canvas_center()
        handled = True

    if "up and left until" in text or "reach the close button" in text or "red button" in text or "closes your mind" in text or "close visual studio code" in text:
        print("Triggering 'walk up-left until close button' event!")
        walk_up_left_until_close(delta=delta)
        close_alignment_triggered = True
        handled = True

    if not close_alignment_triggered:
        for direction, steps in parse_direction_step_commands(text):
            print(f"Triggering '{direction}' event! (steps={steps}, delta={delta}px)")
            move_direction_steps(direction, steps=steps, delta=delta)
            handled = True

    if "click" in text:
        if ENABLE_FINAL_CLOSE_CLICK:
            print("Triggering 'click' event!")
            with mouse_lock:
                click()
        else:
            print("[Story] Click heard but ignored because ENABLE_FINAL_CLOSE_CLICK=False")
        handled = True

    return handled

def audio_callback(indata, frames, time_info, status):
    """Called for each block of audio from the microphone."""
    global is_recording, recorded_audio, silence_counter, rolling_buffer

    audio_block = indata[:, 0].copy()
    rms = np.sqrt(np.mean(audio_block**2))

    if not is_recording:
        # Keep audio in rolling buffer (this captures word beginnings!)
        rolling_buffer.append(audio_block)

        # Check for speech onset
        if rms > VOLUME_THRESHOLD:
            is_recording = True
            silence_counter = 0
            # Include the buffer contents!
            recorded_audio = list(np.concatenate(list(rolling_buffer)))
            recorded_audio.extend(audio_block.tolist())
            print("\r  [Recording...] Speak your sentence!", end="", flush=True)
    else:
        # Currently recording
        recorded_audio.extend(audio_block.tolist())

        if rms < VOLUME_THRESHOLD:
            silence_counter += 1
        else:
            silence_counter = 0

        # Stop recording after silence or max duration
        total_samples = len(recorded_audio)
        if (silence_counter >= silence_blocks and total_samples >= min_samples) or total_samples >= max_samples:
            audio_data = np.array(recorded_audio, dtype=np.float32)

            # Save and transcribe
            sf.write(TEMP_FILE, audio_data, SAMPLE_RATE)

            try:
                result = model.transcribe(TEMP_FILE)
                text = result['text'].strip()

                if text:
                    clear_output(wait=True)
                    print("=" * 50)
                    print("CONTINUOUS SPEECH RECOGNITION")
                    print("=" * 50)
                    print(f"\n  You said: \"{text}\"")
                    print(f"  Language: {result['language']}")
                    duration = len(audio_data) / SAMPLE_RATE
                    print(f"  Duration: {duration:.1f}s")
                    print(f"\nListening... (speak to transcribe)")
                    print("Press the STOP button or Kernel > Interrupt to stop")

                    # Use pitch of voice to control movement speed if you want dynamic steps.
                    # delta = pitch_to_delta(audio_data, SAMPLE_RATE)
                    delta = MOUSE_STEP_PIXELS

                    command_triggered = execute_text_commands(text, delta=delta)
                    if not command_triggered:
                        print("  (No command keyword detected)")

            except Exception as e:
                print(f"\n  Error: {e}")

            # Reset state
            is_recording = False
            recorded_audio = []
            silence_counter = 0
            rolling_buffer.clear()



### Commence the Theatre

In [5]:
# Start listening
print("=" * 50)
print("CONTINUOUS SPEECH RECOGNITION")
print("=" * 50)
print(f"Volume threshold: {VOLUME_THRESHOLD} RMS")
print(f"Silence timeout:  {SILENCE_DURATION}s")
print(f"Max recording:    {MAX_RECORDING}s")
print("\nListening... (speak to transcribe)")
print("Press the STOP button or Kernel > Interrupt to stop")
print("Say 'DARWIN, TAKE OVER' to run the theatre.")
print(f"Story final click armed: {ENABLE_FINAL_CLOSE_CLICK}")

stream = None
try:
    stream = sd.InputStream(
        samplerate=SAMPLE_RATE,
        blocksize=BLOCK_SIZE,
        channels=1,
        callback=audio_callback
    )
    stream.start()

    while True:
        sd.sleep(100)

except KeyboardInterrupt:
    pass
finally:
    if stream is not None:
        stream.stop()
        stream.close()
    if os.path.exists(TEMP_FILE):
        os.remove(TEMP_FILE)
    rolling_buffer.clear()
    is_recording = False
    recorded_audio = []
    silence_counter = 0
    print("\nStopped!")


CONTINUOUS SPEECH RECOGNITION

  You said: "Yeah, he doesn't get my fore"
  Language: en
  Duration: 3.1s

Listening... (speak to transcribe)
Press the STOP button or Kernel > Interrupt to stop
  (No command keyword detected)
  [Recording...] Speak your sentence!
Stopped!
